In [1]:
import time

import pandas as pd

from src.data.load_raw import load_raw_data
from src.data.preprocess import (
    extract_xy,
    merge_features_and_labels,
    standardize_splits,
)
from src.data.split import split_labeled_transactions
from src.evaluation.metrics import calculate_binary_metrics
from src.evaluation.threshold import find_best_f1_threshold
from sklearn.model_selection import TimeSeriesSplit
from src.models.classic import (
    build_dummy_classifier,
    build_logistic_regression,
    build_random_forest,
    build_xgboost,
)

import optuna
from sklearn.model_selection import cross_val_score
import numpy as np

import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=FutureWarning, module="sklearn.linear_model")
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.linear_model")

warnings.filterwarnings("ignore", category=ConvergenceWarning)

C:\Users\pansm\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
features_df, classes_df, edges_df = load_raw_data()

transactions_df = merge_features_and_labels(
    features_df,
    classes_df,
)

splits = split_labeled_transactions(transactions_df)

x_train, y_train = extract_xy(
    splits.train,
    feature_set="all",
)

x_validation, y_validation = extract_xy(
    splits.validation,
    feature_set="all",
)

x_test, y_test = extract_xy(
    splits.test,
    feature_set="all",
)

(
    scaler,
    x_train_scaled,
    x_validation_scaled,
    x_test_scaled,
) = standardize_splits(
    x_train,
    x_validation,
    x_test,
)

In [3]:
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
import numpy as np
import optuna

def objective1(trial, X, y):
    solver = trial.suggest_categorical("solver", ["lbfgs", "liblinear", "saga"])

    if solver == "lbfgs":
        reg_type = trial.suggest_categorical("reg_lbfgs", ["l2", "none"])
        C = np.inf if reg_type == "none" else trial.suggest_float("C_lbfgs", 1e-4, 1e4, log=True)
        l1_ratio = None
    elif solver == "liblinear":
        reg_type = trial.suggest_categorical("reg_lib", ["l1", "l2"])
        C = trial.suggest_float("C_lib", 1e-4, 1e4, log=True)
        l1_ratio = None
    else: # saga
        reg_type = trial.suggest_categorical("reg_saga", ["l1", "l2", "elasticnet", "none"])
        if reg_type == "none":
            C = np.inf
            l1_ratio = None
        else:
            C = trial.suggest_float("C_saga", 1e-4, 1e4, log=True)
            if reg_type == "l1":
                l1_ratio = 1.0
            elif reg_type == "l2":
                l1_ratio = 0.0
            else:
                l1_ratio = trial.suggest_float("l1_ratio_saga", 0.01, 0.99)

    # Crucial addition: Class Weights
    class_weight = trial.suggest_categorical("class_weight", ["balanced", None])
    max_iter = trial.suggest_int("max_iter", 500, 2000, step=500)

    model = build_logistic_regression(
        solver=solver,
        C=C,
        l1_ratio=l1_ratio,
        max_iter=max_iter,
        class_weight=class_weight
    )

    # Fix: TimeSeriesSplit and F1 scoring
    tscv = TimeSeriesSplit(n_splits=5)

    scores = cross_val_score(
        model, X, y, cv=tscv, scoring="f1", n_jobs=-1
    )

    return np.mean(scores)


study = optuna.create_study(direction="maximize", study_name="Logistic_Regression_Tuning")

study.optimize(lambda trial: objective1(trial, x_train, y_train), n_trials=50, show_progress_bar=True)

print("\n--- Optimization Finished ---")
print(f"Best Trial Score: {study.best_value:.4f}")
print("Best Parameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

[I 2026-06-11 23:47:32,135] A new study created in memory with name: Logistic_Regression_Tuning
Best trial: 0. Best value: 0.525324:   2%|▏         | 1/50 [00:04<04:03,  4.96s/it]

[I 2026-06-11 23:47:37,100] Trial 0 finished with value: 0.5253240028659608 and parameters: {'solver': 'lbfgs', 'reg_lbfgs': 'l2', 'C_lbfgs': 0.4925738004382767, 'class_weight': 'balanced', 'max_iter': 500}. Best is trial 0 with value: 0.5253240028659608.


Best trial: 1. Best value: 0.558452:   4%|▍         | 2/50 [01:48<50:35, 63.24s/it]

[I 2026-06-11 23:49:21,129] Trial 1 finished with value: 0.5584516724189156 and parameters: {'solver': 'liblinear', 'reg_lib': 'l2', 'C_lib': 2234.868571356002, 'class_weight': 'balanced', 'max_iter': 500}. Best is trial 1 with value: 0.5584516724189156.


Best trial: 1. Best value: 0.558452:   6%|▌         | 3/50 [01:50<27:34, 35.19s/it]

...

[I 2026-06-11 23:55:25,077] Trial 47 finished with value: 0.6928600311371044 and parameters: {'solver': 'lbfgs', 'reg_lbfgs': 'l2', 'C_lbfgs': 2383.0336547513643, 'class_weight': None, 'max_iter': 1500}. Best is trial 44 with value: 0.694318407023067.


Best trial: 44. Best value: 0.694318:  98%|█████████▊| 49/50 [08:16<00:09,  9.66s/it]

[I 2026-06-11 23:55:49,045] Trial 48 finished with value: 0.5145758599180723 and parameters: {'solver': 'saga', 'reg_saga': 'l1', 'C_saga': 2469.028479491099, 'class_weight': None, 'max_iter': 1000}. Best is trial 44 with value: 0.694318407023067.


Best trial: 44. Best value: 0.694318: 100%|██████████| 50/50 [08:20<00:00, 10.00s/it]

[I 2026-06-11 23:55:52,202] Trial 49 finished with value: 0.5586777651652632 and parameters: {'solver': 'lbfgs', 'reg_lbfgs': 'none', 'class_weight': 'balanced', 'max_iter': 1500}. Best is trial 44 with value: 0.694318407023067.

--- Optimization Finished ---
Best Trial Score: 0.6943
Best Parameters:
  solver: lbfgs
  reg_lbfgs: l2
  C_lbfgs: 9642.769020742975
  class_weight: None
  max_iter: 1500


In [4]:
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
import numpy as np
import optuna

def objective2(trial, X, y):
    n_estimators = trial.suggest_int("n_estimators", 100, 1000, step=100)

    depth_option = trial.suggest_categorical("depth_type", ["unlimited", "constrained"])
    max_depth = None if depth_option == "unlimited" else trial.suggest_int("max_depth", 5, 50)

    min_samples_split = trial.suggest_int("min_samples_split", 2, 50)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2", None])

    # Crucial: Class balancing is heavily needed for the Elliptic dataset
    class_weight = trial.suggest_categorical("class_weight", ["balanced", "balanced_subsample", None])

    model = build_random_forest(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        class_weight=class_weight
    )

    # Use TimeSeriesSplit to prevent temporal data leakage!
    tscv = TimeSeriesSplit(n_splits=5)

    # Score purely on the F1 of the illicit class (assuming it is class 1)
    scores = cross_val_score(
        model, X, y, cv=tscv, scoring="f1", n_jobs=-1
    )

    return np.mean(scores)

# --- Execution ---
study = optuna.create_study(direction="maximize", study_name="Random_Forest_Ultimate_Tuning")

study.optimize(
    lambda trial: objective2(trial, x_train, y_train),
    n_trials=200,
    show_progress_bar=True
)

print("\n--- Optimization Finished ---")
print(f"Best Trial Score: {study.best_value:.4f}")
print("Best Parameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

[I 2026-06-11 23:55:52,272] A new study created in memory with name: Random_Forest_Ultimate_Tuning
Best trial: 0. Best value: 0.781303:   0%|          | 1/200 [00:01<05:36,  1.69s/it]

[I 2026-06-11 23:55:53,962] Trial 0 finished with value: 0.7813030110340113 and parameters: {'n_estimators': 200, 'depth_type': 'constrained', 'max_depth': 16, 'min_samples_split': 28, 'min_samples_leaf': 20, 'max_features': 'log2', 'class_weight': 'balanced_subsample'}. Best is trial 0 with value: 0.7813030110340113.


Best trial: 0. Best value: 0.781303:   1%|          | 2/200 [02:59<5:48:08, 105.50s/it]

[I 2026-06-11 23:58:52,121] Trial 1 finished with value: 0.7157287753509023 and parameters: {'n_estimators': 900, 'depth_type': 'unlimited', 'min_samples_split': 37, 'min_samples_leaf': 4, 'max_features': None, 'class_weight': None}. Best is trial 0 with value: 0.7813030110340113.


Best trial: 0. Best value: 0.781303:   2%|▏         | 3/200 [03:03<3:13:35, 58.96s/it] 

...

[I 2026-06-12 00:27:29,835] Trial 197 finished with value: 0.7976529894760783 and parameters: {'n_estimators': 500, 'depth_type': 'constrained', 'max_depth': 26, 'min_samples_split': 18, 'min_samples_leaf': 11, 'max_features': 'log2', 'class_weight': 'balanced_subsample'}. Best is trial 12 with value: 0.7976529894760783.


Best trial: 12. Best value: 0.797653: 100%|█████████▉| 199/200 [31:40<00:03,  3.38s/it]

[I 2026-06-12 00:27:32,711] Trial 198 finished with value: 0.7922283649169272 and parameters: {'n_estimators': 400, 'depth_type': 'unlimited', 'min_samples_split': 22, 'min_samples_leaf': 12, 'max_features': 'log2', 'class_weight': 'balanced_subsample'}. Best is trial 12 with value: 0.7976529894760783.


Best trial: 12. Best value: 0.797653: 100%|██████████| 200/200 [31:44<00:00,  9.52s/it]

[I 2026-06-12 00:27:37,153] Trial 199 finished with value: 0.7975511726496592 and parameters: {'n_estimators': 600, 'depth_type': 'constrained', 'max_depth': 29, 'min_samples_split': 19, 'min_samples_leaf': 11, 'max_features': 'log2', 'class_weight': 'balanced_subsample'}. Best is trial 12 with value: 0.7976529894760783.

--- Optimization Finished ---
Best Trial Score: 0.7977
Best Parameters:
  n_estimators: 500
  depth_type: unlimited
  min_samples_split: 20
  min_samples_leaf: 11
  max_features: log2
  class_weight: balanced_subsample


In [5]:
def objective3(trial, X, y):
    learning_rate = trial.suggest_float("learning_rate", 1e-3, 0.3, log=True)
    n_estimators = trial.suggest_int("n_estimators", 100, 1000, step=100)
    max_depth = trial.suggest_int("max_depth", 3, 12)
    min_child_weight = trial.suggest_int("min_child_weight", 1, 10)
    subsample = trial.suggest_float("subsample", 0.5, 1.0)
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0)
    gamma = trial.suggest_float("gamma", 0.0, 5.0)
    reg_alpha = trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True)
    reg_lambda = trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True)

    model = build_xgboost(
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_child_weight=min_child_weight,
        gamma=gamma,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        scale_pos_weight=7.634893125361063
    )

    tscv = TimeSeriesSplit(n_splits=5)

    # CHANGED: Optimize for F1, not just recall
    scores = cross_val_score(
        model,
        X,
        y,
        cv=tscv,
        scoring="f1",
        n_jobs=-1
    )

    return np.mean(scores)

# --- Execution ---
study = optuna.create_study(direction="maximize", study_name="XGBoost_Opt")

study.optimize(
    lambda trial: objective3(trial, x_train, y_train),
    n_trials=200,
    show_progress_bar=True
)

print("\n--- Optimization Finished ---")
print(f"Best Trial Score: {study.best_value:.4f}")
print("Best Parameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

[I 2026-06-12 00:27:37,193] A new study created in memory with name: XGBoost_Opt
Best trial: 0. Best value: 0.810358:   0%|          | 1/200 [00:10<33:38, 10.14s/it]

[I 2026-06-12 00:27:47,338] Trial 0 finished with value: 0.810358345003182 and parameters: {'learning_rate': 0.01162998578914489, 'n_estimators': 900, 'max_depth': 10, 'min_child_weight': 2, 'subsample': 0.5730706769131055, 'colsample_bytree': 0.5945025698831565, 'gamma': 2.92160319482429, 'reg_alpha': 2.250480468297573e-08, 'reg_lambda': 0.04143055148660527}. Best is trial 0 with value: 0.810358345003182.


Best trial: 0. Best value: 0.810358:   1%|          | 2/200 [00:18<30:50,  9.34s/it]

[I 2026-06-12 00:27:56,122] Trial 1 finished with value: 0.7024652986757671 and parameters: {'learning_rate': 0.0024586415658123823, 'n_estimators': 600, 'max_depth': 6, 'min_child_weight': 10, 'subsample': 0.8482306566875996, 'colsample_bytree': 0.9081979092927309, 'gamma': 4.039142220184603, 'reg_alpha': 0.001266245687124435, 'reg_lambda': 1.1096064985474062e-07}. Best is trial 0 with value: 0.810358345003182.


Best trial: 0. Best value: 0.810358:   2%|▏         | 3/200 [00:23<24:13,  7.38s/it]

...

[I 2026-06-12 00:43:13,278] Trial 197 finished with value: 0.8845690391607152 and parameters: {'learning_rate': 0.1289896907881066, 'n_estimators': 200, 'max_depth': 7, 'min_child_weight': 2, 'subsample': 0.555768150058604, 'colsample_bytree': 0.6171432789282509, 'gamma': 1.5408081083536536, 'reg_alpha': 3.081778690214963e-06, 'reg_lambda': 1.6592974793652959e-06}. Best is trial 169 with value: 0.8854421313757017.


Best trial: 169. Best value: 0.885442: 100%|█████████▉| 199/200 [15:38<00:02,  2.51s/it]

[I 2026-06-12 00:43:15,741] Trial 198 finished with value: 0.8741769276153855 and parameters: {'learning_rate': 0.13026207052930594, 'n_estimators': 200, 'max_depth': 7, 'min_child_weight': 2, 'subsample': 0.5400301205064678, 'colsample_bytree': 0.6204055680138293, 'gamma': 1.8315546362899517, 'reg_alpha': 5.8921710479678095e-06, 'reg_lambda': 1.3703278915524784e-06}. Best is trial 169 with value: 0.8854421313757017.


Best trial: 169. Best value: 0.885442: 100%|██████████| 200/200 [15:40<00:00,  4.70s/it]

[I 2026-06-12 00:43:18,059] Trial 199 finished with value: 0.8602826394368801 and parameters: {'learning_rate': 0.16711164956709368, 'n_estimators': 200, 'max_depth': 7, 'min_child_weight': 3, 'subsample': 0.5423362137267174, 'colsample_bytree': 0.6176530664114506, 'gamma': 1.8419536732688344, 'reg_alpha': 7.297409390554895e-06, 'reg_lambda': 1.394020928521478e-06}. Best is trial 169 with value: 0.8854421313757017.

--- Optimization Finished ---
Best Trial Score: 0.8854
Best Parameters:
  learning_rate: 0.13619423624369625
  n_estimators: 300
  max_depth: 7
  min_child_weight: 2
  subsample: 0.5656066870561047
  colsample_bytree: 0.5760635388828816
  gamma: 1.6113481494143387
  reg_alpha: 9.764048688630152e-06
  reg_lambda: 1.2107315313358617e-06
